In [1]:
import torch
import torch.nn as nn

# Step 1: Define positional encoding
class LearnablePositionalEncoding(nn.Module):
    def __init__(self, max_len, d_model):
        super().__init__()
        self.pos_embedding = nn.Embedding(max_len, d_model)

    def forward(self, x):
        batch_size, seq_len, d_model = x.size()

        positions = torch.arange(seq_len, device=x.device)
        positions = positions.unsqueeze(0).expand(batch_size, seq_len)

        pos_embed = self.pos_embedding(positions)

        return x + pos_embed


# Step 2: Create dummy input
x = torch.randn(2, 5, 8)  # (batch_size=2, seq_len=5, d_model=8)

# Step 3: Apply positional encoding
pos_encoder = LearnablePositionalEncoding(max_len=10, d_model=8)
output = pos_encoder(x)

# Step 4: Print results
print("Input shape:", x.shape)
print("Output shape:", output.shape)

Input shape: torch.Size([2, 5, 8])
Output shape: torch.Size([2, 5, 8])


In [2]:
print("First input vector:\n", x[0][0])
print("First output vector:\n", output[0][0])

First input vector:
 tensor([-0.3538,  0.7361,  1.8368, -1.2302, -1.4006,  0.5942, -0.4186,  0.0571])
First output vector:
 tensor([-0.7679, -1.1036,  0.4765, -0.4578, -0.4253,  0.4401,  0.4452, -0.1825],
       grad_fn=<SelectBackward0>)


In [3]:
x = torch.randn(2, 5, 8)

In [9]:
class SimpleModel(nn.Module):
    def __init__(self, d_model, max_len):
        super().__init__()

        self.pos_encoder = LearnablePositionalEncoding(max_len, d_model)
        self.fc = nn.Linear(5 * d_model, 1)


    def forward(self, x):
        x = self.pos_encoder(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

In [10]:
model = SimpleModel(d_model=8, max_len=10)

out = model(x)
print("Model output shape:", out.shape)

Model output shape: torch.Size([32, 1])


In [13]:
import torch.optim as optim

# Create model
model = SimpleModel(d_model=8, max_len=10)

# Loss + optimizer
loss_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Dummy data generator
def generate_data(batch_size=32, seq_len=5, d_model=8):
    x = torch.randn(batch_size, seq_len, d_model)
    y = x.sum(dim=1).sum(dim=1, keepdim=True)

    return x, y

# Training loop
x_train, y_train = generate_data()
for epoch in range(200):


    pred = model(x)
    loss = loss_fn(pred, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}")

Epoch 0, Loss: 46.137977600097656
Epoch 20, Loss: 43.52055358886719
Epoch 40, Loss: 41.46641540527344
Epoch 60, Loss: 39.67986297607422
Epoch 80, Loss: 38.001468658447266
Epoch 100, Loss: 36.3952522277832
Epoch 120, Loss: 34.8529052734375
Epoch 140, Loss: 33.37044143676758
Epoch 160, Loss: 31.944971084594727
Epoch 180, Loss: 30.57402801513672
